# **CSC713M (G01) Major Course Output**
### Group: Machine Yearning (Marvin Ivan Mangubat and Jose Carlos Regala)

This is our Jupyter Notebook file documenting our Machine Learning project.

In [99]:
import pandas as pd

# Load the three CSV files from data/raw
amidase = pd.read_csv("data/raw/amidase.csv")
glycosylase = pd.read_csv("data/raw/glycosylase.csv")
peptidase_m15 = pd.read_csv("data/raw/peptidase_m15.csv")
peptidase_m23 = pd.read_csv("data/raw/peptidase_m23.csv")
peptidase = pd.concat([peptidase_m15, peptidase_m23], ignore_index=True) # merge into one list

# check
#print(amidase.head())
#print(glycosylase.head())
print(peptidase.head())

peptidase.dtypes

   Protein ID Protein Name                                   Protein Sequence  \
0          24        107U3  MEKTNTKLKPHWPALSARAFLLPPSTPPQKERTLQKEHALGELPVP...   
1         259        12owM  MGDLTANFNRSEFACKCGCGKDNIKDELAIKVQRVRDLLNRPIRIN...   
2         270        1345B  MSFNTPAVKTYSLRRDGDKQLSPNFRVREFASRDGSDKILICDNLV...   
3         283        137pR  MRMNVNLMNFVDDLLCRNYHFTVTSAFRTEKQNNECGGSPRSQHLV...   
4         387        13qfb  MATQTQVKNFINKIAPIAQEKAKGRDKWSLPSVCIAQACCESGYGT...   

   Cluster Name Representative Accession PhaLP Type  Sequence Length  \
0  phalp2_33672                    107U3  endolysin              214   
1  phalp2_38403                    12owM  endolysin               58   
2  phalp2_30939                    1345B  endolysin              241   
3  phalp2_25037                    137pR  endolysin              100   
4  phalp2_28304                    13qfb  endolysin              314   

                Phage Name Phage Lineage Family Host Names EC Numbers  \
0  Unkn

Protein ID                    int64
Protein Name                    str
Protein Sequence                str
Cluster Name                    str
Representative Accession        str
PhaLP Type                      str
Sequence Length               int64
Phage Name                      str
Phage Lineage Family            str
Host Names                      str
EC Numbers                      str
EC Entry Names              float64
dtype: object

In [100]:
# Assign the class labels
amidase['Label'] = 'Amidase'
glycosylase['Label'] = 'Glycosylase'
peptidase['Label'] = 'Peptidase'


# Merge the CSV files and assign class labels for each of the three.
df_list = [amidase, glycosylase, peptidase]
merged_df = pd.concat(df_list, ignore_index=True)
merged_df.to_csv('data/raw/merged_data.csv', index=False)

# Check the shape and class balance
print("Total dataset shape: ", merged_df.shape)
print("\nClass distribution: ")
print(merged_df['Label'].value_counts())
merged_df.dtypes

Total dataset shape:  (77105, 13)

Class distribution: 
Label
Peptidase      69456
Glycosylase     5077
Amidase         2572
Name: count, dtype: int64


Protein ID                    int64
Protein Name                    str
Protein Sequence                str
Cluster Name                    str
Representative Accession        str
PhaLP Type                      str
Sequence Length               int64
Phage Name                      str
Phage Lineage Family            str
Host Names                      str
EC Numbers                      str
EC Entry Names              float64
Label                           str
dtype: object

Given the massive discrepancy between Peptidase and the other two classes, we need to have some cleaning.

In [101]:
print(merged_df['Protein Sequence'].value_counts())

# Looking at the column names from the csv files, we're looking for the DNA sequences. So, we'll use the column name 'Protein Sequence' and drop any rows where this column is null or missing.
cleaned_df = merged_df.dropna(subset='Protein Sequence')
print("\nTotal dataset shape after dropping missing: ", cleaned_df.shape)

# We also have to ensure that each protein sequence is capitalized to maintain consistency.
cleaned_df['Protein Sequence'] = cleaned_df['Protein Sequence'].str.upper()
print("Total dataset shape after capitalizing: ", cleaned_df.shape)

# Let us also ensure that we only have one copy of each protein sequence to avoid messing with the accuracy of the model later on.
is_unique = cleaned_df['Protein Sequence'].is_unique
if is_unique:
    print("All values are unique.")
else:
    print("The column contains duplicates.")

cleaned_df = cleaned_df.drop_duplicates(subset=['Protein Sequence'], keep='first')
print("Total dataset shape after removing duplicates: ", cleaned_df.shape)

is_unique = cleaned_df['Protein Sequence'].is_unique
if is_unique:
    print("All values are unique.")
else:
    print("The column contains duplicates.")


print(cleaned_df['Protein Sequence'].head())

Protein Sequence
MNIFEMLRIDEGLRLKIYKDTEGYYTIGIGHLLTKSPSLSVAKSELDKAIGRNCNGVITKDEAEKLFNQDVDAAVRGILRNAKLKPVYDSLDAVRRCALINMVFQMGETGVAGFTNSLRMLQQKRWDEAAVNLAKSRWYNQTPNRAKRVIATFRTGTWDAYKNL                                                                                                                                                                                                                                                                                                                            125
MPAYTQDGIAIGIIAEGRRSRSGEGQLDHPVISEKGIVIALAVALVETNLKMYANRSDPESLNFPHDAVGSDANSVGVFQQRAPWWGTVADRMDVARSAAMFYNSLYRQRVGGADYNTDRVSPGTWGQMVQQSAFPDRYDKRMAEARQIYDRLKDRVVGGAPTPPPITAPDPNWRGDPVWLKEVLEAAGLVCHVYDGAYNRGHGDFGEIWGVVAHHTGSFGETPKGIAQHPSLGLASQLYLGRNGEYTLCGVGIAWHAGQGSYPGLPTNDANRLTIGIEAANDGGGSPPGKRDAWSDVQYNAYVRGVAAILRKLGRDSSRVIGHKEWAGTAQGKWDPGGIDMNTFRADVARVMGELGTSTDPVLELLAMPTNQEKLDFIYNEESKKFASRSIYRTPGEGLIDTRAGFVLNVDAMAHQELVDRLAIQYHDSDAIGRIARVAAGQGADPNDTWAKEHALMVLQKIPEEVLKAWQEKNR     98
MQPTLLSKLNQARQL

Given that ESM-2 has a maximum token limit, and that extremely short sequences are not that useful, we must filter the dataframe to only keep rows where the length of the protein sequences is **greater than or equal to 50 AND less than or equal to 1000**.

In [ ]:
# To make this easier, we can use the Sequence Length column to make the filtering easier. But let us first verify if the values of the sequence length column accurately show the length of each amino acid sequence.
cleaned_df['Verify Sequence Length'] = cleaned_df['Protein Sequence'].str.len()
print("The new column's data type is: ", cleaned_df['Verify Sequence Length'].dtypes)
print("The original column's data type is: ", cleaned_df['Sequence Length'].dtypes)

sequence_length_equality = cleaned_df['Sequence Length'].equals(cleaned_df['Verify Sequence Length'])
print("Equality: ", sequence_length_equality)

# Knowing this, we can confidently say that the column has the correct sequence lengths.
cleaned_df = cleaned_df[(cleaned_df['Sequence Length'] >= 50 ) & (cleaned_df['Sequence Length'] <= 1000)]
print("The updated shape is: ", cleaned_df.shape)

The new column's data type is:  int64
The original column's data type is:  int64
Equality:  True
The updated shape is:  (70885, 14)
